In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!git clone https://github.com/Ahmad-techs/fyp-food-classification.git

In [ ]:
%cd fyp-food-classification

In [ ]:
import os
# This will print the path to your dataset
print(os.listdir('/kaggle/input'))

In [ ]:
import os
# This finds where your Food-11 dataset was mounted
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        # Just look for the first few to confirm the structure
        if "Bread" in dirname: # Bread is a subfolder in your Food-11 dataset
            print(f"Dataset found at: {os.path.join(dirname).split('/Bread')[0]}")
            break

In [ ]:
import sys
import torch
from torchvision import transforms
from torch.utils.data import DataLoader

# 1. Add your cloned code folder to system path
sys.path.append('/kaggle/working/fyp-food-classification')

# 2. Imports from your own files
from src.model import CoarseToFineNet
from src.dataset import Food11Dataset
from src.train import train_model

# 3. Path setup
DATA_ROOT = '/kaggle/input/datasets/trolukovich/food11-image-dataset'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 4. Transformations & Loaders
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_ds = Food11Dataset(DATA_ROOT, split='training', transform=transform)
val_ds = Food11Dataset(DATA_ROOT, split='validation', transform=transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)

# 5. Model
model = CoarseToFineNet(num_fine=11, num_coarse=4).to(device)

In [ ]:
# Start the 3-phase training process
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    save_path='/kaggle/working/best_model.pth',
    model_type='single', # Change to 'dual' for your experiments later
    lam=0.0              # Used only if model_type='dual'
)

print("Training finished. Check the file browser on the right to download 'best_model.pth'.")

In [ ]:
import json
import pandas as pd

# 1. Save history to a file so you can download it
with open('baseline_history.json', 'w') as f:
    json.dump(history, f)

# 2. Create a CSV for easy plotting in Excel/Word
df = pd.DataFrame({
    'epoch': range(1, len(history['fine_acc']) + 1),
    'fine_acc': history['fine_acc'],
    'loss': history['loss']
})
df.to_csv('baseline_results.csv', index=False)
print("Files 'baseline_history.json' and 'baseline_results.csv' are ready to download.")

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Use your evaluate function (from your evaluate.py)
# Note: You'll need to pass the model, val_loader, and device
def plot_cm(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels, _ in loader:
            imgs = imgs.to(device)
            out, _ = model(imgs)
            preds = torch.argmax(out, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()

plot_cm(model, val_loader, device)

In [ ]:
# Debugger script
import torch

def inspect_labels(loader):
    max_fine = 0
    max_coarse = 0
    
    for i, (images, fine, coarse) in enumerate(loader):
        if fine.max() > max_fine: max_fine = fine.max()
        if coarse.max() > max_coarse: max_coarse = coarse.max()
        
        if fine.min() < 0 or coarse.min() < 0:
            print("FOUND NEGATIVE LABEL!")
            break
            
    print(f"Max Fine Index found: {max_fine} (Expected < 11)")
    print(f"Max Coarse Index found: {max_coarse} (Expected < 4)")

print("Checking Train Loader...")
inspect_labels(train_loader)

In [ ]:
import importlib
import src.model
import src.train
import src.dataset

# Force reload the modules
importlib.reload(src.model)
importlib.reload(src.train)
importlib.reload(src.dataset)

# Now re-import the classes
from src.model import CoarseToFineNet
from src.train import train_model
from src.dataset import Food11Dataset

# Now re-initialize your model and data...

In [ ]:
import json

# Save the baseline history before you lose it!
with open('/kaggle/working/baseline_history.json', 'w') as f:
    json.dump(history, f)

print("Baseline history saved to baseline_history.json")

In [ ]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

In [ ]:
import os, sys

# Clone the repo fresh
os.chdir('/kaggle/working')
os.system('git clone https://github.com/Ahmad-techs/fyp-food-classification.git')

# Add to path
sys.path.insert(0, '/kaggle/working/fyp-food-classification')

# Confirm it worked
print(os.listdir('/kaggle/working/fyp-food-classification/src'))

In [ ]:
import os, sys

# ── Step 1: Clear any cached failed imports 
for key in list(sys.modules.keys()):
    if 'src' in key or 'model' in key or 'dataset' in key or 'train' in key:
        del sys.modules[key]

# ── Step 2: Add BOTH paths 
SRC    = '/kaggle/working/fyp-food-classification/src'
PARENT = '/kaggle/working/fyp-food-classification'
if SRC    not in sys.path: sys.path.insert(0, SRC)
if PARENT not in sys.path: sys.path.insert(0, PARENT)

# ── Step 3: Fix the broken import inside dataset.py
dataset_file = f'{SRC}/dataset.py'
with open(dataset_file, 'r') as f:
    content = f.read()

if 'from src.coarse_mapping' in content:
    content = content.replace('from src.coarse_mapping', 'from coarse_mapping')
    with open(dataset_file, 'w') as f:
        f.write(content)
    print("Fixed dataset.py import")
else:
    print("dataset.py already fine")

# ── Step 4: Test imports 
from model   import CoarseToFineNet
from train   import train_model
from dataset import Food11Dataset
print("All imports working")

In [ ]:
# Diagnostic — check actual label values coming out of the loader
imgs, fl, cl = next(iter(train_loader))
print(f"Fine labels:   min={fl.min().item()}  max={fl.max().item()}  unique={sorted(fl.unique().tolist())}")
print(f"Coarse labels: min={cl.min().item()}  max={cl.max().item()}  unique={sorted(cl.unique().tolist())}")

In [ ]:
import os, sys, torch
from torchvision import transforms
from torch.utils.data import DataLoader

# Re-clone if needed 
os.chdir('/kaggle/working')
if not os.path.exists('fyp-food-classification'):
    os.system('git clone https://github.com/Ahmad-techs/fyp-food-classification.git')

SRC    = '/kaggle/working/fyp-food-classification/src'
PARENT = '/kaggle/working/fyp-food-classification'
for p in [SRC, PARENT]:
    if p not in sys.path: sys.path.insert(0, p)

#  Fix dataset.py import
with open(f'{SRC}/dataset.py', 'r') as f: content = f.read()
if 'from src.coarse_mapping' in content:
    content = content.replace('from src.coarse_mapping', 'from coarse_mapping')
    with open(f'{SRC}/dataset.py', 'w') as f: f.write(content)

#  Clear import cache 
for key in list(sys.modules.keys()):
    if any(x in key for x in ['src','model','dataset','train','coarse']):
        del sys.modules[key]

from model   import CoarseToFineNet
from train   import train_model
from dataset import Food11Dataset
print("Imports OK")

#  Data loaders — num_workers=0 fixes the worker import issue 
DATA_ROOT = '/kaggle/input/datasets/trolukovich/food11-image-dataset'
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225])
])

train_ds     = Food11Dataset(DATA_ROOT, split='training',   transform=transform)
val_ds       = Food11Dataset(DATA_ROOT, split='validation', transform=transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,
                          num_workers=0)   
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False,
                          num_workers=0)  
print(f'Train: {len(train_ds):,}  Val: {len(val_ds):,}')

#  Verify labels are correct before training 
imgs, fl, cl = next(iter(train_loader))
print(f"Fine labels:   min={fl.min().item()}  max={fl.max().item()}")
print(f"Coarse labels: min={cl.min().item()}  max={cl.max().item()}")
assert cl.max().item() <= 3, "Coarse labels out of range — check coarse_mapping.py"
assert fl.max().item() <= 10, "Fine labels out of range"
print("Label ranges verified  Ready to train.")